In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

# Settings
N_SIMS = 100_000
SEED = 42
DATA_DIR = Path.cwd() / "testfiles_" / "data"
CSV_PATH = DATA_DIR / "test5_3.csv"

# --- Load covariance (first row has column names) ---
df = pd.read_csv(CSV_PATH, header=0)
cols = list(df.columns)
Sigma = df.to_numpy(dtype=float)

# --- Symmetrize (safety) ---
Sigma = (Sigma + Sigma.T) / 2.0
n = Sigma.shape[0]

# --- Higham nearest correlation fix, then scale back to covariance ---
# 1) Convert covariance -> correlation
std = np.sqrt(np.clip(np.diag(Sigma), 0.0, None))
std_safe = np.where(std > 0.0, std, 1.0)  # avoid divide-by-zero for zero-variance dims
D_inv = np.diag(1.0 / std_safe)
C = D_inv @ Sigma @ D_inv
C = (C + C.T) / 2.0

# 2) Higham algorithm (alternating projections) to nearest correlation matrix
def higham_nearest_correlation(A, tol=1e-9, max_iter=100):
    Y = A.copy()
    delta = np.zeros_like(A)
    for _ in range(max_iter):
        R = Y - delta
        # PSD projection by eigenvalue clipping
        w, V = np.linalg.eigh((R + R.T) / 2.0)
        w_clipped = np.clip(w, 0.0, None)
        X = (V * w_clipped) @ V.T
        delta = X - R
        # Unit diagonal projection
        Y_old = Y
        Y = X.copy()
        np.fill_diagonal(Y, 1.0)
        if np.linalg.norm(Y - Y_old, ord='fro') <= tol * np.linalg.norm(Y_old, ord='fro'):
            break
    # ensure symmetry
    return (Y + Y.T) / 2.0

C_fix = higham_nearest_correlation(C)

# 3) Scale back: covariance = D * C_fix * D (use original std, zeros remain zeros)
D = np.diag(std_safe)
Sigma_fix = D @ C_fix @ D
Sigma_fix = (Sigma_fix + Sigma_fix.T) / 2.0

# --- Simulate X ~ N(0, Sigma_fix) using eigen factor (works for PSD) ---
w, V = np.linalg.eigh(Sigma_fix)
w = np.clip(w, 0.0, None)
B = V * np.sqrt(w)  # Sigma_fix = B B^T

rng = np.random.default_rng(SEED)
Z = rng.standard_normal(size=(N_SIMS, n))
X = Z @ B.T

# --- Sample covariance (OUTPUT only) ---
S_hat = np.cov(X, rowvar=False, ddof=1)

print(", ".join(cols))
for i in range(n):
    row = ", ".join(f"{S_hat[i, j]:.16f}" for j in range(n))
    print(row)

x1, x2, x3, x4, x5
0.0851736089630447, 0.0132453685730183, 0.0389443176834606, 0.0083010535470129, 0.0035738127334887
0.0132453685730183, 0.1608523245970108, 0.0535045230154347, 0.0113897127908325, 0.0049202877458407
0.0389443176834606, 0.0535045230154347, 0.0374150314860332, 0.0062346277611498, 0.0026902685918981
0.0083010535470129, 0.0113897127908325, 0.0062346277611498, 0.0016969313974762, 0.0005728953611800
0.0035738127334887, 0.0049202877458407, 0.0026902685918981, 0.0005728953611800, 0.0003154929323933
